# Ticket 5 - Generacion y validacion de respuestas en Colab

Este notebook valida el **Ticket 5** del proyecto **BlueSea AI Assistant**.

Objetivo:

- instalar el proyecto en Google Colab;
- confirmar que existan `chunks.jsonl` y `vectors.jsonl`;
- ejecutar una pregunta con recuperacion RAG y respuesta final citada;
- revisar las fuentes estructuradas usadas por la respuesta;
- demostrar el fallback anti-alucinacion cuando no hay evidencia suficiente.

La logica final vive en `src/rag_bsf/`. Este notebook solo ejecuta y documenta el flujo.

## 1. Clonar el repositorio

Ejecuta esta celda en Google Colab. Si ya estas dentro del repositorio, la celda solo confirma la ruta.

In [ ]:
from pathlib import Path
import os

repo_dir = Path("bluesea-ai-assistant")

if not Path("pyproject.toml").exists():
    if not repo_dir.exists():
        !git clone https://github.com/aantoa/bluesea-ai-assistant.git
    os.chdir(repo_dir)

print("Ruta actual:", Path.cwd())

## 2. Instalar dependencias

`requirements.txt` incluye `-e .`, por eso Colab instala el paquete local desde `src/rag_bsf/` y permite ejecutar `python -m rag_bsf.cli`.

In [ ]:
!python --version
!pip install -r requirements.txt

## 3. Preparar rutas del proyecto

Esta celda evita errores cuando Jupyter o Colab quedan ubicados en otra carpeta. Tambien detiene la ejecucion si detecta una copia en `.Trash`.

In [ ]:
from pathlib import Path
import os
import sys

# Si sabes la ruta real del proyecto, puedes ponerla aqui.
# Si no, dejalo vacio.
PROJECT_ROOT_OVERRIDE = ""

current = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE
    else Path.cwd().resolve()
)

candidates = [
    current,
    current / "repo",
    current.parent,
    current.parent / "repo",
]

project_root = None

for candidate in candidates:
    if (candidate / "pyproject.toml").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise FileNotFoundError(
        f"No se encontro pyproject.toml desde: {current}. "
        "Revisa si estas en la carpeta correcta o si el proyecto esta dentro de repo/."
    )

if ".Trash" in project_root.parts:
    raise RuntimeError(
        f"Estas usando una copia en la papelera: {project_root}. "
        "Abre la carpeta real del proyecto."
    )

os.chdir(project_root)

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

os.environ["PYTHONPATH"] = str(src_path) + os.pathsep + os.environ.get("PYTHONPATH", "")

chunks_path = project_root / "data" / "processed" / "chunks.jsonl"
vectors_path = project_root / "data" / "index" / "vectors.jsonl"
manifest_path = project_root / "data" / "index" / "embeddings_manifest.json"
documents_dir = project_root / "documents"

print("PROJECT_ROOT =", project_root)
print("src agregado =", src_path)
print("documents =", documents_dir)
print("chunks_path =", chunks_path)
print("vectors_path =", vectors_path)
print("manifest_path =", manifest_path)

## 4. Verificar artefactos RAG

El Ticket 5 usa como entrada los artefactos generados por los tickets anteriores:

- `data/processed/chunks.jsonl` del Ticket 2;
- `data/index/vectors.jsonl` del Ticket 3;
- recuperacion de contexto del Ticket 4.

Si los documentos privados no estan versionados, puede ser necesario subirlos antes de regenerar chunks e indice.

In [ ]:
def count_jsonl(path: Path) -> int:
    if not path.exists() or path.stat().st_size == 0:
        return 0
    with path.open(encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())

print("chunks existe:", chunks_path.exists(), "| registros:", count_jsonl(chunks_path))
print("vectors existe:", vectors_path.exists(), "| registros:", count_jsonl(vectors_path))
print("manifest existe:", manifest_path.exists())

## 5. Regenerar artefactos si falta algo

Ejecuta esta celda si `chunks.jsonl` o `vectors.jsonl` no existen. Si no hay documentos fuente en `documents/`, el procesamiento puede generar `0 chunks`.

In [ ]:
if count_jsonl(chunks_path) == 0:
    !python -m rag_bsf.cli process

if count_jsonl(vectors_path) == 0:
    !python -m rag_bsf.cli index

print("chunks:", count_jsonl(chunks_path))
print("vectors:", count_jsonl(vectors_path))

## 6. Ejecutar respuesta citada por CLI

Esta es la validacion principal del Ticket 5: el comando `ask` recupera contexto, genera una respuesta y muestra referencias.

In [ ]:
!python -m rag_bsf.cli ask "cuantos dias de vacaciones tengo" --filter category="Human Resources"

## 7. Inspeccionar respuesta y fuentes estructuradas

La funcion `answer_question()` retorna un objeto con la respuesta, el estado `grounded`, las fuentes y el prompt usado para restringir el modelo al contexto recuperado.

In [ ]:
from rag_bsf.rag_pipeline import answer_question

result = answer_question(
    "cuantos dias de vacaciones tengo",
    top_k=5,
    candidate_k=20,
    metadata_filters={"category": "Human Resources"},
)

print("grounded:", result.grounded)
print("fallback_reason:", result.fallback_reason)
print()
print(result.answer)
print()
print("Fuentes estructuradas:")
for source in result.sources:
    print({
        "label": source.source_label,
        "document_code": source.document_code,
        "filename": source.filename,
        "section": source.section,
        "owner": source.owner,
        "score": round(source.score, 4),
    })

## 8. Revisar el prompt restringido

El prompt instruye al modelo a responder solo con el contexto recuperado y a admitir cuando no encuentra soporte documental.

In [ ]:
print(result.prompt[:1500])

## 9. Demostrar fallback anti-alucinacion

Aqui se fuerza un umbral de confianza muy alto. El agente debe negarse a inventar y devolver un mensaje claro.

In [ ]:
fallback = answer_question(
    "cual es el bono anual exacto de todos los colaboradores",
    top_k=5,
    candidate_k=20,
    min_confidence=999.0,
)

print("grounded:", fallback.grounded)
print("fallback_reason:", fallback.fallback_reason)
print()
print(fallback.answer)

## 10. Criterio de cierre

El Ticket 5 queda validado si:

- `ask` responde usando contexto recuperado;
- la respuesta incluye citas `[S1]`, `[S2]`, etc.;
- las fuentes muestran archivo, seccion y responsable cuando estan disponibles;
- el fallback responde `No encontre esta informacion en los documentos disponibles` cuando no hay soporte suficiente.